# TD Single-Trial Walkthrough: 2 → 2 → 3 Atoms (time-dependent)

The time-dependent counterpart of `single_trial_walkthrough.ipynb`. Where the
original used a parameter `x` in static Hamiltonians, here everything is a
time-dependent $H(v, t)$. We still differentiate w.r.t. a parameter `v`, but
the coefficients now flow through a time symbol `t`, and PSR is routed through
the TD machinery in `td_hamiltonian.py` + `td_psr.py`.

For each case we show:

1. **Hamiltonian** $H(v, t)$ — what we're differentiating
2. **Factoring** — `factor_td_hamiltonian` splits $H(v,t)$ into envelope groups
3. **compile_td** — picks `envelope` (disjoint groups) or `trotter` fallback
4. **PSR-TD branches** — `observable_program_generator_td`, one τ sample
5. **Simulator gradient** — `run_td` on QuTiP sequential runner
6. **Finite-difference reference** — via full TD `sesolve`

### What's different from the TI walkthrough

- **PSR indexes terms, not groups.** Envelope grouping is for compilation.
  PSR still iterates per Pauli product, with `ugrad = ∂f_j/∂v` evaluated at
  each sampled τ — now a *function of τ*, not a constant.
- **Branch segments are mixed.** Segments 0 and 2 are TD (dict form with
  `t_start`, `t_end`); segment 1 (the kick) is the same TI form as before,
  because the kick freezes absolute time (matches Algorithm 1).
- **No hardware ops shown.** `run_td(backend='hardware')` raises
  `NotImplementedError` until `play_wf` and per-group solver calls land in
  `tweezer_mapper`. The simulator path is production-ready.

---
## Setup

In [ ]:
import sys
sys.path.insert(0, "/Users/syue99/research/SimuQ/src/")
sys.path.insert(0, "/Users/syue99/research/SimuQ/differential_computing/")

# Force reload to pick up local changes
to_remove = [k for k in sys.modules if k.startswith('simuq')
             or k in ('td_hamiltonian', 'td_psr', 'qutip_sequential',
                      'observable_program_generator', 'combine_gradient')]
for k in to_remove:
    del sys.modules[k]

import numpy as np
import sympy as sp
import qutip as qp

from simuq import QSystem, Qubit
from simuq.braket.diffQC_provider import diffQCProvider

from td_hamiltonian import (
    factor_td_hamiltonian,
    check_dressing_collision,
    build_channel_envelopes,
)
from td_psr import (
    observable_program_generator_td,
    run_td_sequence,
)

T = np.pi / 2
v_val = 1.0
SEED = 42
t, v = sp.symbols("t v")


def show_groups(groups):
    for i, (env, ti) in enumerate(groups):
        sites = sorted({s for prod, _ in ti.ham for s in prod.keys() if prod[s] != ''})
        terms = [(tuple(prod.to_list()),
                  (float(c) if not isinstance(c, sp.Expr) else c))
                 for prod, c in ti.ham]
        print(f"  group {i}: envelope = {env}  ({len(ti.ham)} term(s), sites={sites})")
        for term in terms:
            print(f"     {term}")


def show_branch(branch):
    for j, seg in enumerate(branch):
        if isinstance(seg, dict) and seg.get('kind') == 'td':
            print(f"  seg {j}: TD  t ∈ [{seg['t_start']:.4f}, {seg['t_end']:.4f}]")
        else:
            H_seg, dur = seg
            terms = [(tuple(p.to_list()), float(c)) for p, c in H_seg.ham]
            print(f"  seg {j}: TI kick  dur={dur:.4f}   Hj = {terms}")


def fd_gradient_td(H_td, t_sym, v_sym, v0, T, psi0, obs, n_q, eps=1e-4):
    def at(v_val):
        H_sub = H_td.set_parameterizedHam({str(v_sym): v_val})
        seg = [{'kind': 'td', 'H': H_sub, 't_sym': t_sym,
                't_start': 0.0, 't_end': T}]
        state = run_td_sequence(seg, psi0, n_q)
        return float(qp.expect(obs, state).real)
    return (at(v0 + eps) - at(v0 - eps)) / (2 * eps)

print("Setup complete.")

---
## Case 1: Single-qubit TD (2-atom register)

**Hamiltonian:** $H(v, t) = v \cdot \sin(t) \cdot Z_0 + \cos(t) \cdot X_0$

Only qubit 0 participates; qubit 1 is idle (the rydberg2d AAIS needs ≥ 2).

**After factoring:** two disjoint groups $\{\sin(t) \cdot Z_0\}$ and
$\{\cos(t) \cdot X_0\}$ — no dressing collision. Strategy: `envelope`.

**PSR term-by-term:** only the $Z_0$ term has $v$ in its coefficient, so one
program entry is emitted, with $\partial f / \partial v = \sin(t)$ evaluated at
the sampled τ. Observable: $\langle Z_0 \rangle$.

In [ ]:
# ── Build H, factor, compile_td ──
qs1 = QSystem(); q1 = [Qubit(qs1) for _ in range(2)]
H1 = v * sp.sin(t) * q1[0].Z + sp.cos(t) * q1[0].X

print("Factoring H1 (with v=1.0 substituted):")
groups1 = factor_td_hamiltonian(H1, t, param_dict={'v': v_val})
show_groups(groups1)
print(f"\ncollision = {check_dressing_collision(groups1)}")

prov1 = diffQCProvider()
tdc1 = prov1.compile_td(H1, t, T, verbose=1)
print(f"\nstrategy = {tdc1['strategy']}")

In [ ]:
# ── PSR-TD branches (single τ sample) ──
np.random.seed(SEED)
tau1 = np.random.rand(1) * T
programs1 = observable_program_generator_td(
    H1, t, T, n_sample=1, n_repetition=1,
    diff_var='v', value=v_val, tau_list=tau1,
)
print(f"τ sample: {tau1}")
print(f"{len(programs1)} term(s) with non-zero ugrad:")
for ti, (branches, ugrad, _) in enumerate(programs1):
    Hj = branches[0][1][0]
    terms = [(tuple(p.to_list()), float(c)) for p, c in Hj.ham]
    print(f"  term {ti}: Hj = {terms}   ugrad(τ) = {ugrad}")

print("\nFirst branch (sgn=-1):")
show_branch(programs1[0][0][0])

In [ ]:
# ── Simulator gradient + FD reference ──
psi0_2 = qp.tensor(qp.basis(2, 0), qp.basis(2, 0))
obs1 = qp.tensor(qp.sigmaz(), qp.qeye(2))

grad_psr1 = prov1.run_td(programs1, obs1, psi0=psi0_2, backend='qutip', verbose=0)
grad_fd1 = fd_gradient_td(H1, t, v, v_val, T, psi0_2, obs1, n_q=2)

print(f"PSR-TD (single trial): {grad_psr1:+.6f}")
print(f"FD  reference:         {grad_fd1:+.6f}")
print(f"sign match: {np.sign(grad_psr1) == np.sign(grad_fd1)}")
print("\n(single-trial PSR has high variance by design — convergence cell below.)")

---
## Case 2: Two-qubit dressing (2-atom, parameter inside envelope)

**Hamiltonian:** $H(v, t) = \sin(v \cdot t) \cdot (Z_0 Z_1 + X_0 + X_1)$

The parameter is *inside* the time envelope — $\partial f / \partial v = t \cos(v t)$,
a genuinely τ-dependent `ugrad`. After substituting $v = 1$ the factored
envelope is $\sin(t)$ and the single group carries three Pauli terms
($Z_0 Z_1$ for dressing, $X_0$ and $X_1$ for Rabi).

**PSR term-by-term:** three terms all share the same envelope, so all three
contribute to the gradient. Observable: $\langle Z_0 Z_1 \rangle$.

In [ ]:
# ── Build H, factor, compile_td ──
qs2 = QSystem(); q2 = [Qubit(qs2) for _ in range(2)]
H2 = sp.sin(v * t) * (q2[0].Z * q2[1].Z + q2[0].X + q2[1].X)

print("Factoring H2 (with v=1.0 substituted):")
groups2 = factor_td_hamiltonian(H2, t, param_dict={'v': v_val})
show_groups(groups2)
print(f"\ncollision = {check_dressing_collision(groups2)}")

prov2 = diffQCProvider()
tdc2 = prov2.compile_td(H2, t, T, verbose=1)
print(f"\nstrategy = {tdc2['strategy']}")

In [ ]:
# ── PSR-TD branches (single τ sample) ──
np.random.seed(SEED)
tau2 = np.random.rand(1) * T
programs2 = observable_program_generator_td(
    H2, t, T, n_sample=1, n_repetition=1,
    diff_var='v', value=v_val, tau_list=tau2,
)
print(f"τ sample: {tau2}")
print(f"{len(programs2)} term(s) with non-zero ugrad:")
for ti, (branches, ugrad, _) in enumerate(programs2):
    Hj = branches[0][1][0]
    terms = [(tuple(p.to_list()), float(c)) for p, c in Hj.ham]
    print(f"  term {ti}: Hj = {terms}   ugrad(τ) = {ugrad}")

print("\nFirst branch (term 0 Z0Z1 kick, sgn=-1):")
show_branch(programs2[0][0][0])

In [ ]:
# ── Simulator gradient + FD reference ──
obs2 = qp.tensor(qp.sigmaz(), qp.sigmaz())
grad_psr2 = prov2.run_td(programs2, obs2, psi0=psi0_2, backend='qutip', verbose=0)
grad_fd2 = fd_gradient_td(H2, t, v, v_val, T, psi0_2, obs2, n_q=2)
print(f"PSR-TD (single trial): {grad_psr2:+.6f}")
print(f"FD  reference:         {grad_fd2:+.6f}")
print(f"sign match: {np.sign(grad_psr2) == np.sign(grad_fd2)}")

---
## Case 3: Three atoms (qubit 2 is a spectator)

**Hamiltonian:** same as Case 2, but embedded in a 3-atom register. Qubit 2
appears in neither $H$ nor the observable, so gradient should equal Case 2's.

In [ ]:
qs3 = QSystem(); q3 = [Qubit(qs3) for _ in range(3)]
H3 = sp.sin(v * t) * (q3[0].Z * q3[1].Z + q3[0].X + q3[1].X)

prov3 = diffQCProvider()
tdc3 = prov3.compile_td(H3, t, T, verbose=1)
print(f"\nstrategy = {tdc3['strategy']}")

np.random.seed(SEED)
tau3 = np.random.rand(1) * T
programs3 = observable_program_generator_td(
    H3, t, T, n_sample=1, n_repetition=1,
    diff_var='v', value=v_val, tau_list=tau3,
)

psi0_3 = qp.tensor(qp.basis(2, 0), qp.basis(2, 0), qp.basis(2, 0))
obs3 = qp.tensor(qp.sigmaz(), qp.sigmaz(), qp.qeye(2))
grad_psr3 = prov3.run_td(programs3, obs3, psi0=psi0_3, backend='qutip')
grad_fd3 = fd_gradient_td(H3, t, v, v_val, T, psi0_3, obs3, n_q=3)
print(f"\nPSR-TD (single trial): {grad_psr3:+.6f}")
print(f"FD  reference:         {grad_fd3:+.6f}")
print(f"sign match: {np.sign(grad_psr3) == np.sign(grad_fd3)}")
print(f"\nSanity: matches Case 2 FD? {abs(grad_fd3 - grad_fd2) < 1e-6}")

---
## Envelope sampling (hardware-waveform preview)

For the hardware path that's still being wired, `build_channel_envelopes`
samples each group's envelope onto an AWG channel with a given weight.
If two groups hit the same channel, their samples add.

Here we pretend Case 1's groups are routed to channels 0 (detuning proxy,
$\sin(t)$) and 1 (Rabi proxy, $\cos(t)$), then plot the samples.

In [ ]:
groups_with_channels = [
    (sp.sin(t), {0: 1.0}),
    (sp.cos(t), {1: 1.0}),
]
sample_rate = 200.0
waveforms = build_channel_envelopes(groups_with_channels, t, T, sample_rate)

for ch, samples in waveforms.items():
    print(f"ch {ch}: {len(samples)} samples, range = [{samples.min():+.3f}, {samples.max():+.3f}]")

try:
    import matplotlib.pyplot as plt
    ts = np.linspace(0, T, len(waveforms[0]), endpoint=False)
    fig, ax = plt.subplots(figsize=(7, 2.5))
    ax.plot(ts, waveforms[0], label='ch 0: sin(t)')
    ax.plot(ts, waveforms[1], label='ch 1: cos(t)')
    ax.set_xlabel('t (μs-equivalent)'); ax.set_ylabel('amplitude'); ax.legend()
    ax.set_title('Case 1 envelope waveforms'); fig.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not available — plot skipped.)')

---
## Convergence: PSR-TD → FD as n_sample grows

Single-trial PSR is unbiased but high-variance. This table walks Case 1 through
growing `n_sample` to show the Monte Carlo error shrinks as expected.

In [ ]:
grad_fd = fd_gradient_td(H1, t, v, v_val, T, psi0_2, obs1, n_q=2)
print(f"FD reference: {grad_fd:+.6f}\n")
print(f"{'n_sample':>10} {'PSR':>12} {'rel_err':>12}")
print('-' * 36)
for n in [1, 10, 100, 600, 2000]:
    np.random.seed(1)
    tau = np.random.rand(n) * T
    progs = observable_program_generator_td(
        H1, t, T, n_sample=n, n_repetition=1,
        diff_var='v', value=v_val, tau_list=tau,
    )
    g = prov1.run_td(progs, obs1, psi0=psi0_2, backend='qutip')
    rel = abs(g - grad_fd) / max(abs(grad_fd), 1e-8)
    print(f"{n:>10d} {g:>+12.6f} {rel:>11.3%}")

---
## Summary

### What the walkthrough shows

| Case | Groups | Strategy | Notes |
|------|--------|----------|-------|
| 1 (2-atom 1-body) | 2 disjoint  | envelope | Only Z₀ term has v; ugrad(τ) = sin(τ) |
| 2 (2-atom 2-body) | 1 group, 3 terms | envelope | Parameter inside envelope; ugrad(τ) = τ·cos(τ) |
| 3 (3-atom 2-body) | 1 group, 3 terms | envelope | Spectator qubit — same gradient as Case 2 |

### Architecture recap

| Component | TI walkthrough | TD walkthrough |
|-----------|---------------|----------------|
| Factoring | (N/A) | `factor_td_hamiltonian` — group by envelope |
| Compile | `compile()` → solver boxes | `compile_td()` → groups + strategy |
| Branches | `[TIHam, dur]` × 3 segs | `[TD-dict, TI kick, TD-dict]` |
| ugrad | single scalar / term | **list per τ / term** (varies with τ) |
| PSR run | `run(backend='qutip')` | `run_td(backend='qutip')` |
| Hardware | ops + ledgers | *NotImplementedError — play_wf TODO* |
| Combine | `combine_gradient_results` | `combine_gradient_results_td` |

### What's still missing (hardware path)

- `play_wf` op in `tweezer_mapper` — time-varying AWG samples
- Per-group solver calls in `compile_td` — so each group gets its own `sol_gvars`
- Cross-group atom-position allocation when multiple envelope groups run concurrently
- `to_pulsedsl` branch emitting `play_wf` samples to the AWG